# Nemotron Reasoning LoRA Solution Writeup

This notebook is structured for Kaggle publication. It reads the pipeline artifacts produced by the repository scripts, renders the required charts and tables, and stays honest when a stage has not been run yet.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def hist_df(items):
    if not items:
        return pd.DataFrame(columns=["bin_start", "bin_end", "count"])
    return pd.DataFrame(items)

download_summary = load_json(ARTIFACTS / "dataset_download_summary.json")
filtering_summary = load_json(ARTIFACTS / "filtering_summary.json")
stage1_summary = load_json(ARTIFACTS / "stage1_sft" / "training_summary.json")
stage1_dataset = load_json(ARTIFACTS / "stage1_sft" / "dataset_summary.json")
stage2_summary = load_json(ARTIFACTS / "stage2_grpo" / "training_summary.json")
eval_summary = load_json(ARTIFACTS / "eval" / "local_eval_summary.json")


## Executive Summary

What was done:

- merged six math reasoning sources with explicit live-ID fallbacks for Hub mismatches
- applied answer extractability, deduplication, length, difficulty, and quality-score filtering
- trained a rank-32 LoRA in two stages: SFT for format alignment and GRPO for answer accuracy plus reasoning control
- evaluated baseline, SFT-only, and SFT+GRPO adapters with deterministic vLLM inference and a best-of-16 ceiling estimate

What worked:

- the data pipeline preserves source provenance and difficulty buckets
- the GRPO stage logs all four requested reward components separately
- the submission packager validates adapter-only artifacts before zipping

Final leaderboard score:

This cell intentionally reads from generated artifacts instead of hard-coding a score. If the score is unavailable, the run has not been completed yet.

In [ ]:
summary_rows = [
    {"stage": "baseline", "validation_accuracy": eval_summary.get("baseline", {}).get("accuracy")},
    {"stage": "SFT-only", "validation_accuracy": eval_summary.get("stage1_sft", {}).get("accuracy")},
    {"stage": "SFT+GRPO", "validation_accuracy": eval_summary.get("stage2_grpo", {}).get("accuracy")},
    {"stage": "best-of-16 ceiling", "validation_accuracy": eval_summary.get("best_of_n", {}).get("accuracy")},
]
display(pd.DataFrame(summary_rows))

## Data Pipeline

Datasets used:

- openai/gsm8k
- lighteval/MATH requested, resolved to HuggingFaceH4/MATH on the live Hub
- AI-MO/NuminaMath-CoT
- nvidia/OpenMathReasoning
- HuggingFaceH4/numina-math-cots requested, resolved to HuggingFaceH4/numina_60k_math_verify_correct_2_4gens_with_rm_scores on the live Hub
- EleutherAI/hendrycks_math

Filtering logic:

- clean answer extractability into `\\boxed{}`
- SHA256 question deduplication with source-priority tie breaking
- reasoning length filter at 200 to 3000 tokens
- difficulty stratification at 20% easy, 40% medium, 40% hard
- quality score = `answer_is_correct * inverse_trace_length`
- keep the top 70% by quality score before final quota selection


In [ ]:
display(pd.DataFrame(download_summary.get("sources", {})).T)
display(pd.DataFrame([{
    "raw_records": filtering_summary.get("raw_records"),
    "deduped_records": filtering_summary.get("deduped_records"),
    "top_quality_records": filtering_summary.get("top_quality_records"),
    "final_selected_records": filtering_summary.get("final_selected_records"),
    "train_records": filtering_summary.get("train_records"),
    "validation_records": filtering_summary.get("validation_records"),
    "grpo_records": filtering_summary.get("grpo_records"),
}]))

before_df = hist_df(filtering_summary.get("quality_histogram_before", []))
after_df = hist_df(filtering_summary.get("quality_histogram_after", []))
fig, ax = plt.subplots(figsize=(10, 4))
if not before_df.empty:
    ax.plot(before_df["bin_start"], before_df["count"], label="before filtering", linewidth=2)
if not after_df.empty:
    ax.plot(after_df["bin_start"], after_df["count"], label="after filtering", linewidth=2)
ax.set_title("Quality Score Distribution Before and After Filtering")
ax.set_xlabel("quality score")
ax.set_ylabel("count")
ax.legend()
plt.show()

## SFT Training

This stage is the format-alignment step. It forces stable structured reasoning and `\\boxed{}` final answers using the requested rank-32 LoRA configuration.

In [ ]:
stage1_log = pd.DataFrame(stage1_summary.get("log_history", []))
display(pd.DataFrame([{
    "backend": stage1_summary.get("backend"),
    "model_id": stage1_summary.get("model_id"),
    "dataset_count": stage1_dataset.get("record_count"),
    "lora_rank": 32,
    "lora_alpha": 64,
    "batch_size_per_device": 2,
    "gradient_accumulation": 4,
    "max_seq_length": 8192,
}]))
if not stage1_log.empty and {"step", "loss"}.issubset(stage1_log.columns):
    stage1_log.dropna(subset=["loss"]).plot(x="step", y="loss", figsize=(10, 4), title="Stage 1 SFT Loss Curve")
    plt.show()
sample_inputs = stage1_dataset.get("sample_inputs", [])[:3]
for idx, sample in enumerate(sample_inputs, start=1):
    print(f"Sample formatted input {idx}\n{'-' * 80}\n{sample[:1500]}\n")

## GRPO Training

Reward design:

- correctness reward
- boxed format reward
- reasoning conciseness reward
- no answer leakage reward

This section doubles as the Best RL Method writeup: it emphasizes the per-component reward behavior instead of just the aggregate reward.

In [ ]:
stage2_log = pd.DataFrame(stage2_summary.get("log_history", []))
reward_columns = [column for column in stage2_log.columns if isinstance(column, str) and column.startswith("reward/") and column.endswith("/mean")]
if not stage2_log.empty and reward_columns:
    stage2_log.plot(x="step", y=reward_columns, figsize=(12, 5), title="GRPO Reward Component Curves")
    plt.show()
if not stage2_log.empty and "reward" in stage2_log.columns:
    stage2_log.plot(x="step", y="reward", figsize=(10, 4), title="Mean Total Reward Trajectory")
    plt.show()
comparison = pd.DataFrame([
    {"model": "baseline", "accuracy": eval_summary.get("baseline", {}).get("accuracy")},
    {"model": "SFT-only", "accuracy": eval_summary.get("stage1_sft", {}).get("accuracy")},
    {"model": "SFT+GRPO", "accuracy": eval_summary.get("stage2_grpo", {}).get("accuracy")},
])
display(comparison)

## Ablation Results

The final prompt variant is selected by validation accuracy, not by preference.

In [ ]:
ablation = pd.DataFrame(eval_summary.get("prompt_ablation", {}).get("variants", []))
display(ablation.sort_values("accuracy", ascending=False) if not ablation.empty else ablation)
eval_summary.get("prompt_ablation", {}).get("best", {})

## Inference Analysis

This section focuses on reasoning-length shifts before and after curation and training.

In [ ]:
before_lengths = hist_df(filtering_summary.get("reasoning_length_histogram_before", []))
after_lengths = hist_df(filtering_summary.get("reasoning_length_histogram_after", []))
fig, ax = plt.subplots(figsize=(10, 4))
if not before_lengths.empty:
    ax.plot(before_lengths["bin_start"], before_lengths["count"], label="before filtering", linewidth=2)
if not after_lengths.empty:
    ax.plot(after_lengths["bin_start"], after_lengths["count"], label="after filtering", linewidth=2)
ax.set_title("Reasoning Trace Length Histogram")
ax.set_xlabel("tokens")
ax.set_ylabel("count")
ax.legend()
plt.show()

## Reproduction Instructions

Install:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
```

Run the pipeline:

```bash
python data/download_datasets.py --streaming
python data/filter_and_curate.py
python data/generate_synthetic.py
python training/stage1_sft.py
python eval/local_eval.py --stage1-dir outputs/stage1_sft
python training/stage2_grpo.py
python eval/local_eval.py --stage1-dir outputs/stage1_sft --stage2-dir outputs/stage2_grpo
python submission/package_lora.py --adapter-dir outputs/stage2_grpo
```

Expected runtime on the target RTX PRO 6000 Blackwell 96 GB machine:

- data download and curation: 1 to 3 hours
- synthetic generation: 4 to 8 hours
- SFT: 8 to 14 hours
- GRPO: 6 to 12 hours
- evaluation and packaging: 1 to 2 hours
- total planned GPU time: 19 to 36 hours

Open Contribution Award emphasis:

- Best Data: quality-score formula plus before/after quality histogram
- Best RL Method: four-component reward design plus reward breakdown curves
- Best Fine-Tuning Method: LoftQ, alpha=`2x rank`, and SFT→GRPO sequencing rationale

Form URL from the user specification:

`https://docs.google.com/forms/d/1O9XN6v0fCDkBVChbbcorBmT95rhca6_ELz8RwwJnx9c/preview`